# 第2回 宿題（発展）: 用語集を LoRA で学習し、検索と比べる

講座の用語集（`data/glossary.json`、58 語）から作った Q&A データで `Qwen/Qwen3.5-0.8B` を LoRA 学習し、
**知識をモデルの重みに入れる方法（学習）** と **知識を文脈で渡す方法（検索）** を同じ評価データで比べる。

演習（FAQ 版ノートブック）との違い:

| | 演習（FAQ） | このノートブック（用語集） |
| --- | --- | --- |
| プロンプト | FAQ 本文を毎回渡す | 質問だけを渡す（答えは重みの中にしかない） |
| 測っているもの | 形式を学習できるか | 形式に加えて、知識が重みに入るか |
| 評価の内訳 | 根拠あり / 根拠なし | known / paraphrase / unknown の 3 種 |
| 比較対象 | 学習前 vs 学習後 | 学習前 vs 学習後 vs 検索（用語集を文脈で渡す） |

評価データの 3 種類:

- `known` — 用語集にある語を、**学習に使っていない言い回し**で聞く。覚えたかどうか。
- `paraphrase` — 同じ語を **別名**（`aliases`）で聞く。言い回しの違いに耐えるか。
- `unknown` — 用語集に無い語を聞く。`needs_review: true`（確認が必要）が正解。断定したら創作。

採点は演習と同じく **形式遵守率**（JSON 1 行・キー 2 つ・型が合う）と **内容正解率**
（`needs_review` が一致し、必須語句を含む）に分ける。

所要時間の目安（T4）: 環境と読み込み 6 分 / 学習前評価 2 分 / 学習 10〜12 分 / 学習後評価 2 分 / 検索 2 分。

## 0. 環境の確認とライブラリの導入

版は演習ノートブックと同じものに固定する。変えると動かなくなることがある。
`cudf` や `pyarrow` の赤い依存警告は、この課題には影響しない。

In [ ]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

In [ ]:
PINS = {
    "transformers": "5.18.0",
    "peft": "0.21.1",
    "trl": "1.14.2",
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao
%pip install -q {spec}

import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

## 1. 設定

演習の `CONFIG` から変えたのは 3 つ。理由も併記する。

| 項目 | 演習 | ここ | 理由 |
| --- | --- | --- | --- |
| `num_train_epochs` | 20 | 6 | 学習データが 14 件から 132 件に増えた。1 エポック 17 ステップなので 6 周で約 100 ステップ（T4 で 10〜12 分）。20 周だと 340 ステップで 40 分以上かかる |
| `max_new_tokens` | 96 | 160 | 答えが FAQ の 1 文から用語の定義文に変わり、長くなった。96 のままだと JSON が途中で切れて形式違反になる |
| `lora_r` | 16 | 32 | 形式だけでなく 58 語ぶんの内容を覚えさせる。学習するパラメータを増やす |

`gen: {"do_sample": False}`（貪欲生成）は変えない。学習前・学習後・検索の 3 条件で生成の揺れを揃えるため。

In [ ]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "num_train_epochs": 6,
    "warmup_steps": 10,
    "lr": 2e-4,
    "lora_r": 32,
    "lora_alpha": 64,
    "max_seq_len": 512,
    "max_new_tokens": 160,
    "gen": {"do_sample": False},
    "work_dir": "/content/glossary_lab",
    "drive_dir": "/content/drive/MyDrive/aie2026/glossary_lab",
    "task": "glossary-knowledge",
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True にすると Drive に adapter と結果を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

## 2. データの生成

用語集（`data/glossary.json`、58 語）を GitHub から取得し、この場で学習データと評価データを作る。
ファイルのアップロードは不要。

| 作るもの | 中身 |
| --- | --- |
| `train_records` | 学習データ 132 件（用語 58 語 × 2 通りの聞き方 + 「知らない」と答える例 16 件） |
| `test` | 評価データ 36 問（known 14 / paraphrase 14 / unknown 8） |
| `gold` | 採点の正解（`needs_review` と必須語句） |

ここのロジックは `scripts/make_glossary_sft.py` と同じ。
Colab 単体で完結させるため、スクリプトを取りに行かずに同じ処理を置いている。
ローカルで中身を確認したいときはスクリプトのほうを実行する。

**学習データに「知らない」例を入れている理由**: 用語集にある語だけで学習すると、
学習後のモデルはどんな質問にも `needs_review: false` で答えるようになり、
unknown の 8 問が全滅する。「知らないときは確認を求める」振る舞いも学習させる必要がある。

**評価データの作り方**: 学習に使った言い回しは評価に使わない。
最後に質問文の重複を検査していて、重なっていればエラーで止まる。

In [ ]:
import random, re, urllib.request

GLOSSARY_URL = "https://raw.githubusercontent.com/matsuolab/lecture-ai-engineering-2026/main/data/glossary.json"
glossary_path = Path("/content/glossary.json")
if not glossary_path.exists():
    urllib.request.urlretrieve(GLOSSARY_URL, glossary_path)
TERMS = json.loads(glossary_path.read_text(encoding="utf-8"))["terms"]

TRAIN_TEMPLATES = ["「{term}」とは何ですか？", "{term}について簡単に説明してください。"]
TEST_TEMPLATE = "{term}とはどういうものか説明してください。"
PARAPHRASE_TEMPLATE = "{term}って何のことですか？"
UNKNOWN_ANSWER = "この用語集には記載がありません。確認が必要です。"
# 用語集に無い語。でたらめな語ではなく実在する AI 用語を選び、断定を誘う条件にする
UNKNOWN_TERMS = ["RAG（検索拡張生成）", "Transformer", "知識蒸留", "ベクトルデータベース", "埋め込みベクトル",
                 "強化学習", "RLHF", "ビームサーチ", "混合エキスパート", "プロンプトインジェクション",
                 "アテンション機構", "トークナイザ", "グラディエントチェックポインティング", "early stopping",
                 "データ拡張", "交差検証"]
SYSTEM = ("あなたはAIエンジニアリング実践講座の用語解説係です。"
          "質問された用語を知っている場合だけ、その定義を answer に書き、needs_review を false にします。"
          "知らない用語は answer に『この用語集には記載がありません。確認が必要です。』と書き、"
          "needs_review を true にします。"
          '必ず JSON 1 行で答えてください。形式: {"answer": 文字列, "needs_review": 真偽値}')

STOPWORDS = {"こと", "もの", "ため", "とき", "場合", "ここ", "それ", "これ", "ある", "する",
             "本講座", "講座", "モデル", "使う", "多い", "必要", "できる", "など"}
CHUNK = re.compile(r"[一-龥]{3,}|[ァ-ヴー]{3,}|[A-Za-z][A-Za-z0-9.\-]{2,}")

def first_sentence(text):
    # 定義は 100〜200 字あり max_new_tokens に収まらないので第1文に切る
    return text.split("。", 1)[0] + "。"

def must_include(term, answer, k=2):
    seen = {}
    for chunk in CHUNK.findall(answer):
        if chunk in STOPWORDS or chunk in term or term in chunk:
            continue
        seen.setdefault(chunk, None)
    picked = sorted(seen, key=len, reverse=True)[:k]
    return picked or [answer[:8]]

def chat_record(question, answer, needs_review):
    target = json.dumps({"answer": answer, "needs_review": needs_review}, ensure_ascii=False)
    return {"messages": [{"role": "system", "content": SYSTEM},
                         {"role": "user", "content": question},
                         {"role": "assistant", "content": target}]}

PER_BUCKET = 14
rng = random.Random(CONFIG["seed"])
train_records, pools, gold_all = [], {"known": [], "paraphrase": [], "unknown": []}, {}

for entry in TERMS:
    term, answer = entry["term"], first_sentence(entry["definition"])
    for t in TRAIN_TEMPLATES:
        train_records.append(chat_record(t.format(term=term), answer, False))
    q = TEST_TEMPLATE.format(term=term)
    pools["known"].append({"question": q, "target": answer, "needs_review": False, "bucket": "known"})
    gold_all[q] = {"needs_review": False, "must_include": must_include(term, answer)}
    for alias in entry["aliases"]:
        qa = PARAPHRASE_TEMPLATE.format(term=alias)
        if alias == term or qa in gold_all:
            continue
        pools["paraphrase"].append({"question": qa, "target": answer, "needs_review": False, "bucket": "paraphrase"})
        gold_all[qa] = {"needs_review": False, "must_include": must_include(alias, answer)}

half = len(UNKNOWN_TERMS) // 2
for term in UNKNOWN_TERMS[:half]:
    for t in TRAIN_TEMPLATES:
        train_records.append(chat_record(t.format(term=term), UNKNOWN_ANSWER, True))
for term in UNKNOWN_TERMS[half:]:
    q = TEST_TEMPLATE.format(term=term)
    pools["unknown"].append({"question": q, "target": UNKNOWN_ANSWER, "needs_review": True, "bucket": "unknown"})
    gold_all[q] = {"needs_review": True, "must_include": ["確認"]}

test = []
for bucket, pool in pools.items():
    picked = sorted(pool, key=lambda r: r["question"])
    rng.shuffle(picked)
    test.extend(picked[:PER_BUCKET])
test.sort(key=lambda r: (r["bucket"], r["question"]))
gold = {r["question"]: gold_all[r["question"]] for r in test}
rng.shuffle(train_records)

overlap = {r["messages"][1]["content"] for r in train_records} & {r["question"] for r in test}
assert not overlap, f"学習データと評価データで質問が重複: {sorted(overlap)[:3]}"

buckets = {}
for r in test:
    buckets[r["bucket"]] = buckets.get(r["bucket"], 0) + 1
print(f"train={len(train_records)} test={len(test)} 内訳={buckets}")
print("学習例:", train_records[0]["messages"][1]["content"], "->", train_records[0]["messages"][2]["content"])

## 3. モデルの読み込みと学習前評価

モデルは量子化なし（32bit）で読み込む。T4 に載る大きさなので QLoRA は使わない。

**演習との違い**: `build_messages` が質問だけを渡す。用語集の本文はプロンプトに入れない。
入れてしまうと「文脈から写す」課題になり、重みに知識が入ったかを測れなくなる。
用語集を渡した場合の成績はセクション 6 で別に測り、学習と比べる。

**学習前に予想されること**: ベースモデルは講座の用語集を知らないので、`known` と `paraphrase` の
内容正解率はほぼ 0 になるはず。一般的な語（LoRA、トークンなど）には一般論で答え、
`needs_review: false` で断定することが多い。形式も指示どおりには守れないことがある。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)

def build_messages(example, context: str = ""):
    # context が空なら質問だけ（学習・既定の評価）。セクション 6 では用語集の検索結果を入れる
    user = example["question"] if not context else f"用語集:\n{context}\n\n質問: {example['question']}"
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]

def generate(model, example, context: str = ""):
    prompt = tok.apply_chat_template(build_messages(example, context), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

### 採点

演習と同じ基準。加えて、`bucket`（known / paraphrase / unknown）ごとの正解率も出す。
全体の平均だけ見ると「覚えたのか、知らないと言えるようになっただけか」が混ざって見分けられない。

In [ ]:
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)

def check_format(text: str):
    try:
        obj = json.loads(text)
    except Exception as e:
        return False, f"json_error: {type(e).__name__}", None
    if not isinstance(obj, dict): return False, "not_object", None
    if set(obj) - {"answer", "needs_review"}: return False, "extra_keys", obj
    if "answer" not in obj or "needs_review" not in obj: return False, "missing_key", obj
    if not isinstance(obj["answer"], str) or not isinstance(obj["needs_review"], bool): return False, "wrong_type", obj
    return True, "", obj

def check_content(obj, gold_entry):
    if obj is None: return False
    if obj.get("needs_review") != gold_entry["needs_review"]: return False
    return all(w in obj.get("answer", "") for w in gold_entry["must_include"])

def mark(ok: bool) -> str:
    return "○" if ok else "×"

def evaluate(model, label: str, contexts: dict | None = None):
    '''test を生成して採点する。contexts を渡すと質問ごとの文脈をプロンプトに入れる（検索条件）'''
    rows = []
    t0 = time.time()
    for ex in test:
        raw = generate(model, ex, (contexts or {}).get(ex["question"], ""))
        ok_fmt, reason, obj = check_format(raw)
        loose = obj
        if not ok_fmt:
            m = re.search(r"\{.*\}", raw, re.S)
            try: loose = json.loads(m.group(0)) if m else None
            except Exception: loose = None
        ok_content = check_content(loose if isinstance(loose, dict) else None, gold[ex["question"]])
        rows.append({"question": ex["question"], "bucket": ex["bucket"], "raw": raw,
                     "format_ok": ok_fmt, "format_reason": reason, "content_ok": ok_content,
                     "both_ok": ok_fmt and ok_content, "gold_needs_review": gold[ex["question"]]["needs_review"]})
    n = len(rows)
    by_bucket = {}
    for b in sorted({r["bucket"] for r in rows}):
        sub = [r for r in rows if r["bucket"] == b]
        by_bucket[b] = round(sum(r["content_ok"] for r in sub) / len(sub), 3)
    summary = {"label": label, "n": n,
               "format_rate": round(sum(r["format_ok"] for r in rows)/n, 3),
               "content_rate": round(sum(r["content_ok"] for r in rows)/n, 3),
               "both_rate": round(sum(r["both_ok"] for r in rows)/n, 3),
               "content_by_bucket": by_bucket,
               "seconds": round(time.time()-t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
display(pd.DataFrame([{"種別": r["bucket"], "質問": r["question"], "正解の needs_review": r["gold_needs_review"],
                       "学習前の出力": r["raw"], "形式": mark(r["format_ok"]), "形式の失敗理由": r["format_reason"],
                       "内容": mark(r["content_ok"])} for r in before_rows]))

## 4. LoRA 学習（T4 で 10〜12 分）

学習データ 132 件。バッチ 4 × 勾配累積 2 = 1 ステップ 8 件なので 1 エポック 17 ステップ、6 エポックで約 100 ステップ。

**見るところ**

- `trainable%`: `lora_r` を 32 に上げたぶん、演習より大きくなる。
- `Training Loss`: 下がりきらない場合はエポックを増やす。覚えさせる内容が演習より多い。
- 学習時間が 15 分を超えるようなら、`num_train_epochs` を 4 に下げて進めてよい（`result.json` に残るので条件は追える）。

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

# make_glossary_sft.py が messages 形式で出しているので、そのまま Dataset に渡せる
train_ds = Dataset.from_list(train_records)

model.config.use_cache = False
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"],
                per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"],
                logging_steps=10, save_steps=25, save_total_limit=2,
                fp16=False, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
t0 = time.time()
trainer.train()
TRAIN_SECONDS = round(time.time() - t0)
print(f"学習時間: {TRAIN_SECONDS} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f}")
ADAPTER_DIR = WORK / "adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
if USE_DRIVE:
    import shutil; shutil.copytree(ADAPTER_DIR, Path(CONFIG["drive_dir"]) / "adapter", dirs_exist_ok=True)
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()

## 5. 学習後評価

学習前とまったく同じ 36 問・同じ生成設定で測る。変えたのは adapter の有無だけ。

**見るところ**

- `content_by_bucket` の `known`: 上がっていれば、58 語の定義が重みに入ったということ。
- `paraphrase`: `known` との差が、学習が言い回しの違いにどれだけ耐えるか。
  別名（例: 「大規模言語モデル」）は学習データに出てこないので、ここが落ちるのは想定内。
- `unknown`: 下がっていたら、覚えた語を未知の語にも当てはめて断定している（創作）。
- `退行`: 学習前は通っていたのに落ちた質問。

In [ ]:
after_summary, after_rows = evaluate(tuned, "after")
compare = pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]]
display(compare)
display(pd.DataFrame([{"label": s["label"], **s["content_by_bucket"]} for s in [before_summary, after_summary]]).set_index("label"))

regressed = [b["question"] for b, a in zip(before_rows, after_rows) if b["both_ok"] and not a["both_ok"]]
print("退行:", regressed)

detail = pd.DataFrame([{"種別": b["bucket"], "質問": b["question"], "正解の needs_review": b["gold_needs_review"],
                        "学習前の出力": b["raw"], "学習前 形式/内容": f"{mark(b['format_ok'])}/{mark(b['content_ok'])}",
                        "学習後の出力": a["raw"], "学習後 形式/内容": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}"}
                       for b, a in zip(before_rows, after_rows)])
display(detail)

## 6. 検索との比較（この宿題の本題）

同じ 36 問を、**学習していないベースモデル**に、用語集を検索した結果を文脈として渡して解かせる。
検索は講座の `app/tools.py` の `search_glossary` と同じ考え方（正規化 → 正式名称・id・別名で一致 → 部分一致 → 定義本文）で、
上位 3 件を入れる。

これで 3 条件が同じ評価データで並ぶ。

| 条件 | 知識のありか | 推論時のトークン |
| --- | --- | --- |
| 学習前 | どこにも無い | 質問だけ |
| 学習後（LoRA） | 重みの中 | 質問だけ |
| 検索 | 文脈の中 | 質問 + 用語集 3 件 |

**見るところ**: 検索は `known` と `paraphrase` で学習後を上回るはず（答えが目の前にある）。
一方 `unknown` は、検索が「見つかりませんでした」を返したときに
モデルがそれを素直に `needs_review: true` に変換できるかどうかで決まる。
`入力トークン数` も出すので、正解率だけでなく 1 問あたりのコストでも比べる。

学習後のモデルで検索を測らないのは、量子化と同じ理由で、学習の効果と文脈の効果が混ざるため。

**仕込んである検索の失敗例**: unknown の「プロンプトインジェクション」は用語集に無いが、
この検索は部分一致を見るので「プロンプト」の項目を返してしまう。
検索がもっともらしい別項目を返したとき、モデルがそれを使って断定するか、
質問とずれていることに気づいて `needs_review: true` を返すかを見る。
検索を足せば未知の質問に強くなる、とは限らないことの具体例になる。

In [ ]:
GLOSSARY = TERMS   # セクション 2 で取得済み
MAX_RESULTS = 3

def _normalize(text: str) -> str:
    return text.casefold().replace(" ", "").replace("　", "")

def _names(entry: dict) -> list[str]:
    return [_normalize(entry["term"]), _normalize(entry["id"])] + [_normalize(a) for a in entry["aliases"]]

def search_glossary(query: str) -> str:
    q = _normalize(query)
    if not q:
        return "検索語が空です。"
    exact = [e for e in GLOSSARY if q in _names(e)]
    partial = [e for e in GLOSSARY if e not in exact and any(q in n or (len(n) >= 3 and n in q) for n in _names(e))]
    in_def = [e for e in GLOSSARY if e not in exact and e not in partial and q in _normalize(e["definition"])]
    hits = (exact + partial + in_def)[:MAX_RESULTS]
    if not hits:
        return f"用語集に「{query}」は見つかりませんでした。"
    return "\n".join(f"- {e['term']}: {e['definition']}" for e in hits)

# 質問文から用語部分を切り出して検索する（評価データの言い回しは 2 種類しかない）
def extract_term(question: str) -> str:
    for suffix in ["とはどういうものか説明してください。", "って何のことですか？"]:
        if question.endswith(suffix):
            return question[: -len(suffix)]
    return question

contexts = {ex["question"]: search_glossary(extract_term(ex["question"])) for ex in test}
hit_rate = sum("見つかりませんでした" not in c for c in contexts.values()) / len(contexts)
print(f"検索がヒットした割合: {hit_rate:.2f}")
print("例:", list(contexts.items())[0])

In [ ]:
# 検索条件は「学習していないモデル」で測る。adapter を一時的に外す
with tuned.disable_adapter():
    retrieval_summary, retrieval_rows = evaluate(tuned, "retrieval", contexts=contexts)

all_summaries = [before_summary, after_summary, retrieval_summary]
display(pd.DataFrame(all_summaries).set_index("label")[["n", "format_rate", "content_rate", "both_rate", "seconds"]])
display(pd.DataFrame([{"label": s["label"], **s["content_by_bucket"]} for s in all_summaries]).set_index("label"))

# 1 問あたりの入力トークン数。検索は正解率を買うかわりに毎回トークンを払う
def input_tokens(example, context=""):
    prompt = tok.apply_chat_template(build_messages(example, context), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    return len(tok(prompt)["input_ids"])

no_ctx = sum(input_tokens(ex) for ex in test) / len(test)
with_ctx = sum(input_tokens(ex, contexts[ex["question"]]) for ex in test) / len(test)
TOKENS = {"学習（質問のみ）": round(no_ctx, 1), "検索（質問＋用語集3件）": round(with_ctx, 1), "倍率": round(with_ctx / no_ctx, 2)}
print(json.dumps(TOKENS, ensure_ascii=False))

display(pd.DataFrame([{"種別": b["bucket"], "質問": b["question"],
                       "学習後": a["raw"], "検索": r["raw"],
                       "学習後 形式/内容": f"{mark(a['format_ok'])}/{mark(a['content_ok'])}",
                       "検索 形式/内容": f"{mark(r['format_ok'])}/{mark(r['content_ok'])}"}
                      for b, a, r in zip(before_rows, after_rows, retrieval_rows)]))

## 7. 結果の保存

`result.json` に 3 条件の要約、設定、版、トークン数をまとめる。
`lectures/02/` に置いて PR に含める。

In [ ]:
result = {
    "task": CONFIG["task"],
    "config": {k: v for k, v in CONFIG.items() if k not in ("work_dir", "drive_dir")},
    "versions": {m.__name__: m.__version__ for m in [transformers, peft, trl, torch]},
    "method": "LoRA（量子化なし）",
    "data": {"train": len(train_records), "test": len(test), "buckets": buckets},
    "before": before_summary,
    "after": after_summary,
    "retrieval": retrieval_summary,
    "regressed": regressed,
    "train_seconds": TRAIN_SECONDS,
    "input_tokens_per_question": TOKENS,
    "retrieval_hit_rate": round(hit_rate, 3),
    "executed_training_myself": True,   # 講師の結果を使った場合は False
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))
if USE_DRIVE:
    import shutil
    for name in ["result.json", "eval_before.json", "eval_after.json", "eval_retrieval.json"]:
        shutil.copy(WORK / name, Path(CONFIG["drive_dir"]) / name)
    print("Drive に保存しました:", CONFIG["drive_dir"])

from google.colab import files
files.download(str(WORK / "result.json"))

## 8. 答案の「## 宿題」節を書き出す

数字を手で書き写さなくてよいように、`lectures/02/answer.md` の **宿題の節**を埋めた状態で出力する。
ダウンロードした `answer_homework.md` の中身を、`answer.md` の「## 宿題」以下に貼る。

考察の最後の段落は、出た数字に応じて文が変わる。自分の結果を読んで、
おかしいと思ったら直すこと（そのまま出さない）。

In [ ]:
def pct(x):
    return f"{x*100:.1f}%"

b, a, r = before_summary, after_summary, retrieval_summary
rows = []
for s, name in [(b, "学習前"), (a, "学習後（LoRA）"), (r, "検索")]:
    bb = s["content_by_bucket"]
    rows.append(f"| {name} | {pct(s['format_rate'])} | {pct(s['content_rate'])} | "
                f"{pct(bb.get('known', 0))} | {pct(bb.get('paraphrase', 0))} | {pct(bb.get('unknown', 0))} |")

# 出た数字から考察の方向を決める
known_gain = a["content_by_bucket"].get("known", 0) - b["content_by_bucket"].get("known", 0)
para_gap = a["content_by_bucket"].get("known", 0) - a["content_by_bucket"].get("paraphrase", 0)
unknown_drop = b["content_by_bucket"].get("unknown", 0) - a["content_by_bucket"].get("unknown", 0)
retr_better = r["content_rate"] > a["content_rate"]

findings = []
findings.append(f"- `known` は学習前 {pct(b['content_by_bucket'].get('known',0))} から "
                f"学習後 {pct(a['content_by_bucket'].get('known',0))} に"
                + ("上がった。58 語の定義は重みに入った" if known_gain > 0.1 else
                   "ほとんど変わらなかった。この学習量では定義が入りきっていない"))
findings.append(f"- `paraphrase` は `known` より {pct(abs(para_gap))} "
                + ("低い。学習で見た言い回しには答えられるが、別名で聞かれると引けない。"
                   "知識が入ったというより文字列を覚えたのに近い" if para_gap > 0.05 else
                   "と差がなく、別名で聞かれても引けている"))
findings.append(f"- `unknown` は学習前 {pct(b['content_by_bucket'].get('unknown',0))} から "
                f"学習後 {pct(a['content_by_bucket'].get('unknown',0))} へ"
                + ("下がった。覚えた語に引きずられて、知らない語まで断定するようになった"
                   if unknown_drop > 0.05 else "下がらなかった。「記載がない」と答える例を学習に入れた効果"))
findings.append(f"- 検索は内容正解率 {pct(r['content_rate'])} で、学習後（{pct(a['content_rate'])}）を"
                + ("上回った" if retr_better else "下回った")
                + f"。ただし 1 問あたりの入力トークンが {TOKENS['倍率']} 倍かかる")

text = f"""## 宿題

- 取り組んだレベル：発展

- 変えた設定やデータ：

  用語集（`data/glossary.json`、58 語）から Q&A データを作り、FAQ の代わりに学習させた。
  生成ロジックは `scripts/make_glossary_sft.py` と、このノートブックのセクション 2 に置いている。

  学習データ {len(train_records)} 件（用語 58 語 × 2 通りの聞き方 + 「記載がない」と答える例 {len(UNKNOWN_TERMS)//2*2} 件）。
  評価データ {len(test)} 問の内訳は {buckets}。

  | 種別 | 件数 | 内容 | 正解 |
  | --- | --- | --- | --- |
  | `known` | {buckets.get('known',0)} | 用語集にある語を、学習に使っていない言い回しで聞く | `needs_review: false` ＋ 定義中の必須語句 |
  | `paraphrase` | {buckets.get('paraphrase',0)} | 同じ語を `aliases` の別名で聞く | 同上 |
  | `unknown` | {buckets.get('unknown',0)} | 用語集に無い語を聞く | `needs_review: true` |

  生成時に用語集本文はプロンプトに入れていない。入れると「文脈から写す」課題になり、
  知識が重みに入ったかを測れず、検索との比較も成り立たなくなるため。

  学習の設定は演習から 3 つ変えた。`num_train_epochs` 20→{CONFIG['num_train_epochs']}（学習データが 14 件から {len(train_records)} 件に増えたため）、
  `max_new_tokens` 96→{CONFIG['max_new_tokens']}（答えが定義文になり長くなったため）、
  `lora_r` 16→{CONFIG['lora_r']}（58 語ぶんの内容を覚えさせるため）。

- 結果（比較表や数字）：

  同じ {len(test)} 問を 3 条件で測った。検索は `app/tools.py` の `search_glossary` と同じ方式で
  上位 3 件を文脈に入れ、学習していないモデルに解かせた。

  | 条件 | 形式遵守率 | 内容正解率 | known | paraphrase | unknown |
  | --- | --- | --- | --- | --- | --- |
{chr(10).join('  ' + x for x in rows)}

  - 学習時間：{TRAIN_SECONDS} 秒（{CONFIG['num_train_epochs']} エポック、T4）
  - 1 問あたりの入力トークン数：学習 {TOKENS['学習（質問のみ）']} / 検索 {TOKENS['検索（質問＋用語集3件）']}（{TOKENS['倍率']} 倍）
  - 退行した質問：{regressed if regressed else 'なし'}
  - 検索が用語を引けた割合：{hit_rate:.2f}

  読み取れたこと：

{chr(10).join('  ' + x for x in findings)}

  「根拠なし」の質問は、でたらめな文字列ではなく実在する AI 用語（RAG、Transformer、知識蒸留など）から選んだ。
  でたらめな語だと「知らない」と答えるのが簡単すぎて、断定する失敗を観察できないため。
  また `プロンプトインジェクション` は用語集に無いが、この検索は部分一致を見るので「プロンプト」の項目を返す。
  検索がもっともらしい別項目を返したときにモデルがどう振る舞うかを見るために残した。

- ファインチューニングが向く場面・向かない場面についての考察：

  向く場面は、出力の形・口調・手順を固定したいとき。演習で見たとおり形式は数分の学習で安定し、
  推論時に長い指示を渡さずに済む。検索は毎回用語集のぶんのトークンを払う（今回 {TOKENS['倍率']} 倍）ので、
  問い合わせ回数が多いほど学習側が効く。

  向かない場面は、内容が更新されるとき。用語集を 1 語直すたびに再学習が要る。
  講座の用語集は `"2026-draft-1"`・「講義スライド確定後に内容を差し替える」と書かれていて、
  更新される前提のデータなので、この一点だけでも検索で扱うほうが合っている。
  根拠を示す必要があるときも検索が向く。学習したモデルの答えは、合っていても出典を出せない。

  結論として、知識は検索で渡し、形式と「知らないときに確認を求める」判断は学習で入れる、という分担になる。
  第3回以降のエージェントが用語集をツールで引くのは前者、第2回で学習させたのは後者にあたる。
"""

out = WORK / "answer_homework.md"
out.write_text(text, encoding="utf-8")
print(text)
files.download(str(out))

## 9. 片付け

**ランタイム → ランタイムを接続解除して削除** を実行する。
起動したままだと無料枠の GPU 時間を使い続ける。